# CYK Parser Success Rate Evaluation

This notebook evaluates the CYK parser success rate across all California committee hearings in the corpus.

In [1]:
from pathlib import Path
import subprocess
import zipfile

# This is where the unzipped corpus file is stored
CORPUS_FILE_PATH = 'DH2024_Corpus_Release/'
corpus_dir = Path(CORPUS_FILE_PATH)
zip_path = Path("digitaldemocracy-2015-2018/DH2024_Corpus_Release.zip")
repo_dir = Path("digitaldemocracy-2015-2018")

# Clone repository if not present
if not repo_dir.is_dir():
    print("Corpus directory not found. Cloning repository...")
    subprocess.run(
        ["git", "clone", "https://huggingface.co/datasets/iatpp/digitaldemocracy-2015-2018"],
        check=True,
    )
    print("Repository cloned successfully.")

# Extract zip file if corpus directory doesn't exist
if not corpus_dir.is_dir():
    if zip_path.exists():
        print(f"Extracting {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('.')
        print("Extraction complete.")
    else:
        print(f"Error: {zip_path} not found!")
else:
    print(f"Corpus already extracted at {corpus_dir}")

Corpus already extracted at DH2024_Corpus_Release


## Load California Hearings

In [2]:
from src import HearingLoader, HearingTagger
from src.grammar.Parser import Parser
import time

# Initialize the HearingLoader with the corpus path
loader = HearingLoader(corpus_path='DH2024_Corpus_Release/')
tagger = HearingTagger()
parser = Parser()

print("Loading all committee hearings...")
start_time = time.time()
all_hearings = loader.load_all_committee_hearings()
load_time = time.time() - start_time
print(f"Total hearings loaded: {len(all_hearings)} (took {load_time:.1f}s)")

# Filter for California hearings only
ca_hearings = [h for h in all_hearings if h.state == 'CA']
print(f"California hearings: {len(ca_hearings)}")

# Filter out hearings with no bill discussed
ca_hearings_with_bills = [h for h in ca_hearings if h.bid != 'CA_NO BILL DISCUSSED'][:500]
print(f"California hearings with bills: {len(ca_hearings_with_bills)}")

Loading all committee hearings...
invalid literal for int() with base 10: 'hid'
invalid literal for int() with base 10: 'hid'
invalid literal for int() with base 10: 'hid'
Total hearings loaded: 8218 (took 10.5s)
California hearings: 8218
California hearings with bills: 500


## Run Tagger and Evaluate CYK Parser (Combined)

To improve efficiency, we'll tag and parse in a single loop with progress tracking.

In [3]:
import numpy as np

print("Processing hearings (tagging + parsing)...")
print("This will take several minutes...\n")

parse_results = []
successful_parses = []
failed_parses = []
tagging_errors = []

total = len(ca_hearings_with_bills)
start_time = time.time()

for i, hearing in enumerate(ca_hearings_with_bills):
    # Progress update every 50 hearings
    if (i + 1) % 100 == 0:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        remaining = (total - i - 1) / rate if rate > 0 else 0
        print(f"Processed {i + 1}/{total} hearings ({(i+1)/total*100:.1f}%)")
    
    try:
        # Tag the hearing
        tagged_hearing = tagger(hearing)
        
        # Try to parse
        try:
            parse_trees = list(parser.get_all_parses_as_nltk_trees(tagged_hearing, max_parses=1) or [])
            parse_success = len(parse_trees) > 0
            
            parse_results.append(int(parse_success))
            
            if parse_success:
                successful_parses.append((hearing.hid, hearing.bid, len(parse_trees)))
            else:
                failed_parses.append((hearing.hid, hearing.bid))
                
        except Exception as e:
            # Parsing failed with exception
            parse_results.append(0)
            failed_parses.append((hearing.hid, hearing.bid))
            
    except Exception as e:
        # Tagging failed
        tagging_errors.append((hearing.hid, hearing.bid, str(e)))
        continue

total_time = time.time() - start_time
print(f"\nProcessing complete! Total time: {total_time:.1f}s ({total_time/60:.1f} minutes)")
print(f"Successfully processed: {len(parse_results)}/{total} hearings")
print(f"Tagging errors: {len(tagging_errors)}")

Processing hearings (tagging + parsing)...
This will take several minutes...

Processed 100/500 hearings (20.0%)
Processed 200/500 hearings (40.0%)
Processed 300/500 hearings (60.0%)
Processed 400/500 hearings (80.0%)
Processed 500/500 hearings (100.0%)

Processing complete! Total time: 83.2s (1.4 minutes)
Successfully processed: 497/500 hearings
Tagging errors: 3


In [4]:
tagging_errors

[(52363, 'CA_201720180AB1246', 'min() iterable argument is empty'),
 (254780, 'CA_201720180AB2970', '100929'),
 (255466, 'CA_201720180SB910', 'min() iterable argument is empty')]

## Results

In [5]:
total_hearings = len(parse_results)
successful_count = np.sum(parse_results)
failed_count = total_hearings - successful_count
success_rate = np.mean(parse_results) * 100 if parse_results else 0

print("=" * 80)
print("CYK Parser Statistics for 2015-2018 California Hearings")
print("=" * 80)
print(f"\nTotal hearings evaluated: {total_hearings}")
print(f"Successful parses:\t{successful_count}")
print(f"Failed parses:\t{failed_count}")
print(f"\nSuccess rate:\t{success_rate:.2f}%")
print("\n" + "=" * 80)

CYK Parser Statistics for 2015-2018 California Hearings

Total hearings evaluated: 497
Successful parses:	444
Failed parses:	53

Success rate:	89.34%



## Detailed Statistics

In [6]:
successful_parses[:5]

[(52054, 'CA_201720180AB92', 1),
 (52054, 'CA_201720180AB816', 1),
 (52054, 'CA_201720180AB547', 1),
 (52054, 'CA_201720180AB822', 1),
 (52054, 'CA_201720180AB677', 1)]

In [7]:
print("\nSuccessful Parses Statistics:")
print(f"  Total: {len(successful_parses)}")

if successful_parses:
    parse_tree_counts = [count for _, _, count in successful_parses]
    print(f"  Average parse trees per hearing: {np.mean(parse_tree_counts):.2f}")
    print(f"  Min parse trees: {np.min(parse_tree_counts)}")
    print(f"  Max parse trees: {np.max(parse_tree_counts)}")
    
    # Show distribution of parse tree counts
    unique, counts = np.unique(parse_tree_counts, return_counts=True)
    print("\n  Parse tree count distribution:")
    for tree_count, freq in zip(unique, counts):
        print(f"    {tree_count} tree(s): {freq} hearings ({freq/len(successful_parses)*100:.1f}%)")

print(f"\nFailed Parses:")
print(f"  Total: {len(failed_parses)}")

# Show first 10 failed parses as examples
if failed_parses:
    print("\n  Examples of failed parses (first 10):")
    for i, (hid, bid) in enumerate(failed_parses[:10]):
        print(f"    {i+1}. Hearing ID: {hid}, Bill: {bid}")

if tagging_errors:
    print(f"\nTagging Errors:")
    print(f"  Total: {len(tagging_errors)}")
    print("\n  Examples (first 5):")
    for i, (hid, bid, error) in enumerate(tagging_errors[:5]):
        print(f"    {i+1}. Hearing ID: {hid}, Bill: {bid}")
        print(f"       Error: {error[:100]}..." if len(error) > 100 else f"       Error: {error}")


Successful Parses Statistics:
  Total: 444
  Average parse trees per hearing: 1.00
  Min parse trees: 1
  Max parse trees: 1

  Parse tree count distribution:
    1 tree(s): 444 hearings (100.0%)

Failed Parses:
  Total: 53

  Examples of failed parses (first 10):
    1. Hearing ID: 52363, Bill: CA_201720180AB1223
    2. Hearing ID: 52773, Bill: CA_201720180AB1635
    3. Hearing ID: 54013, Bill: CA_201720180SJR6
    4. Hearing ID: 254735, Bill: CA_201720180AB2361
    5. Hearing ID: 52364, Bill: CA_201720180AB550
    6. Hearing ID: 52364, Bill: CA_201720180AB275
    7. Hearing ID: 52997, Bill: CA_201720180AB1335
    8. Hearing ID: 52997, Bill: CA_201720180AB519
    9. Hearing ID: 253000, Bill: CA_201720180AB1500
    10. Hearing ID: 254741, Bill: CA_201720180AB2324

Tagging Errors:
  Total: 3

  Examples (first 5):
    1. Hearing ID: 52363, Bill: CA_201720180AB1246
       Error: min() iterable argument is empty
    2. Hearing ID: 254780, Bill: CA_201720180AB2970
       Error: 100929
   

# Failure Cases

In [8]:
# Load failed hearing examples for analysis
import random

random.seed(42)

# Sample 5 failed hearings for detailed analysis
sampled_failed_hids = random.sample(failed_parses, min(5, len(failed_parses)))
print(f"Sampled {len(sampled_failed_hids)} failed hearings for analysis:")
for hid, bid in sampled_failed_hids:
    print(f"  - Hearing ID: {hid}, Bill: {bid}")

# Load the full hearings
failed_hearing_objects = []
for hid, bid in sampled_failed_hids:
    # Find the hearing in our loaded data
    for h in ca_hearings_with_bills:
        if h.hid == hid and h.bid == bid:
            # Tag it
            try:
                tagged = tagger(h)
                if tagged:
                    failed_hearing_objects.append(tagged)
                    break
            except Exception as e:
                print(f"Error tagging {hid}/{bid}: {e}")
                break

print(f"\nSuccessfully loaded {len(failed_hearing_objects)} failed hearings")

Sampled 5 failed hearings for analysis:
  - Hearing ID: 254662, Bill: CA_201720180AB2001
  - Hearing ID: 52997, Bill: CA_201720180AB519
  - Hearing ID: 52773, Bill: CA_201720180AB1635
  - Hearing ID: 255541, Bill: CA_201720180SB1375
  - Hearing ID: 52296, Bill: CA_201720180AB225
Error tagging 254662/CA_201720180AB2001: 100929
Error tagging 255541/CA_201720180SB1375: min() iterable argument is empty
Error tagging 52296/CA_201720180AB225: min() iterable argument is empty

Successfully loaded 2 failed hearings


In [9]:
# Export failed hearings to a text file
output_text_file = "failed_parse_analysis.txt"

with open(output_text_file, 'w', encoding='utf-8') as f:
    f.write("Failed Parse Analysis\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"Total failed hearings: {len(failed_parses)}\n")
    f.write(f"Sample exported: {len(failed_hearing_objects[:10])}\n\n")
    
    for i, hearing in enumerate(failed_hearing_objects[:10]):  # Export first 10 detailed examples
        f.write(f"{'='*80}\n")
        f.write(f"Failed Hearing {i+1}\n")
        f.write(f"{'='*80}\n\n")
        
        f.write(f"Bill: {hearing.bid}\n")
        f.write(f"Hearing ID: {hearing.hid}\n")
        f.write(f"Committee: {hearing.cid}\n")
        
        # Token sequence
        tokens = parser.tokenizer.tokenize_utterances(hearing)
        token_names = [token[0].name for token in tokens]
        f.write(f"Token Sequence ({len(tokens)} tokens):\n")
        f.write("  " + " -> ".join(token_names) + "\n\n")
        
        # Full transcript
        f.write(f"\nFull Transcript:\n")
        f.write(str(hearing))
        f.write("\n\n")

print(f"Failed hearing analysis exported to: {output_text_file}")

Failed hearing analysis exported to: failed_parse_analysis.txt


## Compare Successful vs Failed Hearings

Let's compare the characteristics of successful vs failed parses to identify what makes a hearing unparseable.

In [10]:
from collections import Counter

# Analyze all failed hearings (not just the sample)
print("Analyzing all failed hearing patterns...")
print("=" * 80)

# Get all failed hearings
all_failed_hearings = []
for hid, bid in failed_parses:
    for h in ca_hearings_with_bills:
        if h.hid == hid and h.bid == bid:
            try:
                tagged = tagger(h)
                if tagged:
                    all_failed_hearings.append(tagged)
                    break
            except Exception as e:
                # Skip hearings that fail to tag
                break

print(f"Successfully loaded {len(all_failed_hearings)} failed hearings for pattern analysis\n")

# Analyze token sequence lengths
failed_token_lengths = []
failed_token_sequences = []
failed_section_counts = []

for hearing in all_failed_hearings:
    tokens = parser.tokenizer.tokenize_utterances(hearing)
    token_names = tuple([token[0].name for token in tokens])
    
    failed_token_lengths.append(len(tokens))
    failed_token_sequences.append(token_names)

# Token length statistics
print("Token Length Statistics for Failed Hearings:")
print(f"  Mean: {np.mean(failed_token_lengths):.1f}")
print(f"  Median: {np.median(failed_token_lengths):.1f}")
print(f"  Min: {np.min(failed_token_lengths)}")
print(f"  Max: {np.max(failed_token_lengths)}")

# Most common token sequences in failures
print("\nMost Common Token Sequences in Failed Hearings (top 10):")
sequence_counter = Counter(failed_token_sequences)
for i, (seq, count) in enumerate(sequence_counter.most_common(10), 1):
    print(f"  {i}. [{' -> '.join(seq)}] ({count} occurrences)")

# Most common starting tokens
starting_tokens = [seq[0] if seq else None for seq in failed_token_sequences]
print("\nMost Common Starting Tokens in Failed Hearings:")
starting_counter = Counter(starting_tokens)
for token, count in starting_counter.most_common():
    if token:
        print(f"  {token}: {count} ({count/len(starting_tokens)*100:.1f}%)")

# Most common ending tokens
ending_tokens = [seq[-1] if seq else None for seq in failed_token_sequences]
print("\nMost Common Ending Tokens in Failed Hearings:")
ending_counter = Counter(ending_tokens)
for token, count in ending_counter.most_common():
    if token:
        print(f"  {token}: {count} ({count/len(ending_tokens)*100:.1f}%)")

Analyzing all failed hearing patterns...
Successfully loaded 34 failed hearings for pattern analysis

Token Length Statistics for Failed Hearings:
  Mean: 31.6
  Median: 25.0
  Min: 3
  Max: 109

Most Common Token Sequences in Failed Hearings (top 10):
  1. [BILL_AUTHOR -> PUBLIC -> PRESIDING_CHAIR -> PUBLIC -> PUBLIC -> PUBLIC -> PRESIDING_CHAIR -> PUBLIC -> PRESIDING_CHAIR -> BILL_AUTHOR -> PRESIDING_CHAIR -> SECRETARY -> PRESIDING_CHAIR -> BILL_AUTHOR -> PRESIDING_CHAIR -> PRESIDING_CHAIR -> PRESIDING_CHAIR -> SECRETARY -> PRESIDING_CHAIR -> PRESIDING_CHAIR -> SECRETARY -> PRESIDING_CHAIR -> SECRETARY -> PRESIDING_CHAIR -> PRESIDING_CHAIR -> COMMITTEE_MEMBER -> PUBLIC] (1 occurrences)
  2. [NONLEGISLATOR -> PRESIDING_CHAIR -> SECRETARY -> PRESIDING_CHAIR -> PRESIDING_CHAIR -> COMMITTEE_MEMBER -> PRESIDING_CHAIR -> BILL_AUTHOR -> SECRETARY -> PRESIDING_CHAIR -> SECRETARY -> PRESIDING_CHAIR] (1 occurrences)
  3. [PRESIDING_CHAIR -> BILL_AUTHOR -> PRESIDING_CHAIR -> EXPERT -> PRESIDING

## Visualize Section Speaker Rule Requirements

Compare failed hearings against the grammar's section speaker position requirements.

In [11]:
# Analyze token sequences of failed hearings
print("Failed Hearing Token Sequences")
print("=" * 80)

for i, hearing in enumerate(failed_hearing_objects):
    print(f"\n{'='*80}")
    print(f"Failed Hearing {i+1}")
    print(f"{'='*80}")
    print(f"Bill: {hearing.bid}")
    print(f"Hearing ID: {hearing.hid}")
    
    # Get tokens
    tokens = parser.tokenizer.tokenize_utterances(hearing)
    
    # Show token sequence
    print(f"\nToken count: {len(tokens)}")
    print("\nToken sequence (SpeakerPositionEnum):")
    token_names = [token[0].name for token in tokens]
    print("  " + " -> ".join(token_names))

Failed Hearing Token Sequences

Failed Hearing 1
Bill: CA_201720180AB519
Hearing ID: 52997

Token count: 54

Token sequence (SpeakerPositionEnum):
  PRESIDING_CHAIR -> SECRETARY -> PRESIDING_CHAIR -> SECRETARY -> PRESIDING_CHAIR -> SECRETARY -> PRESIDING_CHAIR -> SECRETARY -> COMMITTEE_MEMBER -> PRESIDING_CHAIR -> BILL_AUTHOR -> PRESIDING_CHAIR -> BILL_AUTHOR -> EXPERT -> PRESIDING_CHAIR -> PUBLIC -> PRESIDING_CHAIR -> PUBLIC -> PRESIDING_CHAIR -> PUBLIC -> PUBLIC -> PRESIDING_CHAIR -> PUBLIC -> PRESIDING_CHAIR -> PUBLIC -> PUBLIC -> PRESIDING_CHAIR -> PUBLIC -> PRESIDING_CHAIR -> COMMITTEE_MEMBER -> EXPERT -> COMMITTEE_MEMBER -> EXPERT -> COMMITTEE_MEMBER -> EXPERT -> COMMITTEE_MEMBER -> EXPERT -> PRESIDING_CHAIR -> BILL_AUTHOR -> PRESIDING_CHAIR -> SECRETARY -> PRESIDING_CHAIR -> SECRETARY -> PRESIDING_CHAIR -> PRESIDING_CHAIR -> SECRETARY -> COMMITTEE_MEMBER -> SECRETARY -> COMMITTEE_MEMBER -> SECRETARY -> PRESIDING_CHAIR -> BILL_AUTHOR -> EXPERT -> PRESIDING_CHAIR

Failed Hearing 2